In [ ]:
from langgraph.graph import MessagesState
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, RemoveMessage
from langgraph.graph import StateGraph, START
from langgraph.checkpoint.memory import InMemorySaver

In [ ]:
load_dotenv()

True

In [ ]:
model = ChatGroq(model="llama-3.3-70b-versatile")

In [ ]:
class ChatState(MessagesState):
    summary: str

In [ ]:
def summarize_conversation(state:ChatState):
    existing_summary=state['summary']
    if existing_summary:
        prompt=(
             f"Existing summary:\n{existing_summary}\n\n"
            "Extend the summary using the new conversation above."
        )
    else:
        prompt=("summarize the conversation above")

    message_for_summary=state["messages"]+[
        HumanMessage(content=prompt)
    ]

    response=model.invoke(message_for_summary)
    

    messages_to_delete = state["messages"][:-2]
    
    return {
        "summary":response.content,
        "messages":[RemoveMessage(id=m.id) for m in messages_to_delete],
        }
    

In [ ]:
def chat_node(state:ChatState):
    messages=[]
    if state["summary"]:
        messages.append({
            "role":"system",
            "content":f"Conversation summary:\n{state['summary']}"
        })

    messages.extend(state["messages"])

    print(state["messages"])
    # print(messages)
    response=model.invoke(messages)
    return {"messages":[response]}


In [ ]:
def should_summarize(state: ChatState):
    return len(state["messages"]) > 6

In [ ]:
builder = StateGraph(ChatState)

builder.add_node("chat", chat_node)
builder.add_node("summarize", summarize_conversation)

builder.add_edge(START,"chat")
builder.add_conditional_edges(
    "chat",
    should_summarize,
    {
        True:'summarize',
        False:'__end__',
    }
)
builder.add_edge("summarize", "__end__")

In [ ]:
builder

In [ ]:
checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

In [ ]:
config = {"configurable": {"thread_id": "t1"}}

def run_turn(text: str):
    out = graph.invoke({"messages": [HumanMessage(content=text)], "summary": ""}, config=config)
    return out

In [ ]:
# gives the current version of the state
def show_state():
    snap = graph.get_state(config)
    vals = snap.values
    print("\n--- STATE ---")
    print("summary:", vals.get("summary", ""))
    print("num_messages:", len(vals.get("messages", [])))
    print("messages:")
    for m in vals.get("messages", []):
        print("-", type(m).__name__, ":", m.content[:80])

In [ ]:
run_turn('Quantum Physics')
show_state()

[HumanMessage(content='Quantum Physics', additional_kwargs={}, response_metadata={}, id='24194b76-dc4d-4cc2-a5d5-b02e36f9ef57')]



--- STATE ---
summary: 
num_messages: 2
messages:
- HumanMessage : Quantum Physics
- AIMessage : Quantum physics, also known as quantum mechanics, is a branch of physics that st


In [ ]:
run_turn('How is Albert Einstien related?')
show_state()

[HumanMessage(content='Quantum Physics', additional_kwargs={}, response_metadata={}, id='24194b76-dc4d-4cc2-a5d5-b02e36f9ef57'), AIMessage(content='Quantum physics, also known as quantum mechanics, is a branch of physics that studies the behavior of matter and energy at an atomic and subatomic level. It is a fundamental theory that describes the behavior of particles and systems that are too small to be explained by classical physics.\n\n**Key Principles:**\n\n1. **Wave-Particle Duality**: Quantum objects, such as electrons, can exhibit both wave-like and particle-like behavior depending on how they are observed.\n2. **Uncertainty Principle**: It is impossible to know certain properties of a quantum object, such as its position and momentum, simultaneously with infinite precision.\n3. **Superposition**: Quantum objects can exist in multiple states simultaneously, which is known as a superposition of states.\n4. **Entanglement**: Quantum objects can become "entangled" in such a way that

In [ ]:
run_turn('What are some of Einstien"s fampus work')
show_state()

[HumanMessage(content='Quantum Physics', additional_kwargs={}, response_metadata={}, id='24194b76-dc4d-4cc2-a5d5-b02e36f9ef57'), AIMessage(content='Quantum physics, also known as quantum mechanics, is a branch of physics that studies the behavior of matter and energy at an atomic and subatomic level. It is a fundamental theory that describes the behavior of particles and systems that are too small to be explained by classical physics.\n\n**Key Principles:**\n\n1. **Wave-Particle Duality**: Quantum objects, such as electrons, can exhibit both wave-like and particle-like behavior depending on how they are observed.\n2. **Uncertainty Principle**: It is impossible to know certain properties of a quantum object, such as its position and momentum, simultaneously with infinite precision.\n3. **Superposition**: Quantum objects can exist in multiple states simultaneously, which is known as a superposition of states.\n4. **Entanglement**: Quantum objects can become "entangled" in such a way that

In [ ]:
run_turn('Explain special theory of relativity')
show_state()

[HumanMessage(content='Quantum Physics', additional_kwargs={}, response_metadata={}, id='24194b76-dc4d-4cc2-a5d5-b02e36f9ef57'), AIMessage(content='Quantum physics, also known as quantum mechanics, is a branch of physics that studies the behavior of matter and energy at an atomic and subatomic level. It is a fundamental theory that describes the behavior of particles and systems that are too small to be explained by classical physics.\n\n**Key Principles:**\n\n1. **Wave-Particle Duality**: Quantum objects, such as electrons, can exhibit both wave-like and particle-like behavior depending on how they are observed.\n2. **Uncertainty Principle**: It is impossible to know certain properties of a quantum object, such as its position and momentum, simultaneously with infinite precision.\n3. **Superposition**: Quantum objects can exist in multiple states simultaneously, which is known as a superposition of states.\n4. **Entanglement**: Quantum objects can become "entangled" in such a way that